# Día 9 · Unidad 10 — Estadística aplicada (I): tendencia central, variabilidad y correlación con SciPy

**Objetivos de aprendizaje**
- Usar `numpy` y `scipy.stats` (no solo `pandas.describe()`) para calcular medidas de tendencia central y variabilidad.
- Conocer las variantes de la media (ponderada, armónica, geométrica) y cuándo cada una es la que corresponde.
- Calcular varianza, desviación estándar, asimetría (*skewness*) y percentiles con la sintaxis correcta de NumPy/SciPy.
- Calcular e interpretar tres coeficientes de correlación (Pearson, Spearman, Kendall) con `scipy.stats`.

**Duración estimada:** 75-90 minutos.

**Requisitos previos:** conocimientos previos de estadística descriptiva (media, varianza, correlación) — este notebook no os va a explicar *qué* es una desviación estándar, sino *cómo se calcula en Python*. Del Día 9: ninguno (arranca la Unidad 10, independiente de la Unidad 9 de la mañana).

**Nota:** este notebook usa datos reales del curso (`data/raw/cartera_polizas.csv`, `data/raw/siniestros.csv`) en vez de los datos sueltos de ejemplo del material original.


## 0. Por qué SciPy y no solo `pandas.describe()`

`DataFrame.describe()` os da de un vistazo media, desviación estándar y cuartiles — perfecto para una primera exploración. Pero para el resto de esta unidad (medias no aritméticas, asimetría con corrección de sesgo, tres tipos de correlación con su p-valor, y en los próximos notebooks distribuciones de probabilidad y pruebas de hipótesis) necesitáis `scipy.stats`, que va mucho más allá de lo que cubre `pandas` por sí solo.

> 📊 En Excel esto sería la diferencia entre el resumen automático de una tabla dinámica (`describe()`) y usar funciones estadísticas específicas como `COEF.DE.CORREL()`, `PRUEBA.T()` o `DISTR.NORM.N()` — `scipy.stats` es, en esencia, la caja de herramientas estadísticas de Python equivalente a esas funciones de Excel, con bastante más profundidad.


In [1]:
import math
import numpy as np
import pandas as pd
import scipy.stats

cartera = pd.read_csv("../data/raw/cartera_polizas.csv")
siniestros = pd.read_csv("../data/raw/siniestros.csv")

primas = cartera["prima_anual"].to_numpy()
importes_siniestros = siniestros["importe_pagado"].to_numpy()

print(f"{len(primas)} primas anuales | {len(importes_siniestros)} importes de siniestro")


3000 primas anuales | 1200 importes de siniestro


## 1. Medidas de tendencia central

### Media aritmética

Ya la conocéis. En Python: `np.mean(array)` o el método `.mean()`.


In [2]:
print(f"Prima media de la cartera: {np.mean(primas):.2f} €")
print(f"Prima media (método): {primas.mean():.2f} €")


Prima media de la cartera: 411.24 €
Prima media (método): 411.24 €


`np.mean()` devuelve `nan` si hay algún valor `nan` en el array — a diferencia de `pandas`, que los ignora por defecto en sus agregaciones. Si necesitáis ignorarlos explícitamente en NumPy, usad `np.nanmean()`.


In [3]:
primas_con_nan = np.append(primas[:5], np.nan)
print("Con np.mean():   ", np.mean(primas_con_nan))
print("Con np.nanmean():", np.nanmean(primas_con_nan))


Con np.mean():    nan
Con np.nanmean(): 413.58000000000004


> ⚠️ **Error típico**: calcular `np.mean()` sobre un array con nulos sin darse cuenta, y que el resultado sea silenciosamente `nan` en vez de un número — a diferencia de una fórmula de Excel mal protegida, que suele dar un error visible, `np.mean()` con `nan` de por medio "funciona" sin lanzar ninguna excepción.

### Media ponderada

Cuando cada observación no pesa lo mismo. Ejemplo de dominio: la prima media por **producto**, ponderada por el número de pólizas de cada producto (en vez de la media simple de las primas medias de cada producto, que trataría por igual a AUTO —con 1361 pólizas— y a VIDA —con 429—).


In [4]:
prima_media_por_producto = cartera.groupby("producto")["prima_anual"].mean()
num_polizas_por_producto = cartera.groupby("producto").size()

media_simple_de_medias = prima_media_por_producto.mean()  # trata cada producto por igual
media_ponderada = np.average(prima_media_por_producto, weights=num_polizas_por_producto)

print(f"Media simple de las medias por producto:    {media_simple_de_medias:.2f} €")
print(f"Media ponderada por nº de pólizas (correcta): {media_ponderada:.2f} €")
print(f"Media directa de toda la cartera (control):   {cartera['prima_anual'].mean():.2f} €")


Media simple de las medias por producto:    457.13 €
Media ponderada por nº de pólizas (correcta): 411.24 €
Media directa de toda la cartera (control):   411.24 €


Fijaos en que la media ponderada por número de pólizas coincide (salvo redondeo) con la media directa sobre toda la cartera — es la comprobación numérica que valida que la ponderación está bien planteada.

### Media armónica y media geométrica

- **Media armónica** (`scipy.stats.hmean`): más sensible a valores pequeños, útil para promediar tasas o ratios (p. ej. velocidades, rendimientos).
- **Media geométrica** (`scipy.stats.gmean`): apropiada para promediar tasas de crecimiento o variables en escalas muy distintas.


In [5]:
franquicias = cartera.loc[cartera["franquicia"] > 0, "franquicia"].to_numpy()  # excluye 0 (VIDA/SALUD)

print(f"Media aritmética de franquicias:  {np.mean(franquicias):.2f} €")
print(f"Media armónica de franquicias:    {scipy.stats.hmean(franquicias):.2f} €")
print(f"Media geométrica de franquicias:  {scipy.stats.gmean(franquicias):.2f} €")


Media aritmética de franquicias:  328.38 €
Media armónica de franquicias:    254.20 €
Media geométrica de franquicias:  289.45 €


Como es habitual, armónica ≤ geométrica ≤ aritmética — cuanto más dispersos estén los datos, mayor la diferencia entre las tres.

### Mediana y moda


In [6]:
print(f"Mediana de la prima anual: {np.median(primas):.2f} €")
print(f"Media de la prima anual:    {np.mean(primas):.2f} €")

# La mediana de importes de siniestro está muy por debajo de la media: cola larga a la derecha
# (pocos siniestros muy grandes tiran de la media hacia arriba, algo típico en severidad de siniestros)
print(f"\nMediana importe siniestro: {np.median(importes_siniestros):.2f} €")
print(f"Media importe siniestro:   {np.mean(importes_siniestros):.2f} €")


Mediana de la prima anual: 378.65 €
Media de la prima anual:    411.24 €

Mediana importe siniestro: 703.99 €
Media importe siniestro:   2200.55 €


In [7]:
# scipy.stats.mode() exige datos numéricos (a partir de SciPy 1.11 ya no acepta texto/categorías).
# Para una columna numérica sí es directo -- p. ej. la edad de tomador más frecuente:
moda_edad = scipy.stats.mode(cartera["edad_tomador"].to_numpy(), keepdims=False)
print(f"Edad de tomador más frecuente: {moda_edad.mode} años ({moda_edad.count} pólizas)")

# Para una columna de texto/categoría (como "producto"), la moda se obtiene con pandas, no con SciPy:
print(f"Producto más frecuente: {cartera['producto'].mode().iloc[0]} "
      f"({cartera['producto'].value_counts().iloc[0]} pólizas)")


Edad de tomador más frecuente: 80 años (61 pólizas)
Producto más frecuente: AUTO (1361 pólizas)


## 2. Medidas de variabilidad

### Varianza y desviación estándar

> ⚠️ **Error típico**: olvidar `ddof=1`. Por defecto, `np.var()`/`np.std()` calculan la varianza **poblacional** (dividiendo entre 𝑛). Cuando trabajáis con una **muestra** — que es casi siempre el caso con datos reales de cartera — hay que pedir explícitamente `ddof=1` (divide entre 𝑛−1, la corrección de Bessel) para no subestimar la varianza real de la población.


In [8]:
print(f"Desviación estándar (muestral, ddof=1): {np.std(primas, ddof=1):.2f} €")
print(f"Desviación estándar (poblacional, ddof=0): {np.std(primas, ddof=0):.2f} €")  # ligeramente distinta
print(f"Varianza muestral:                          {np.var(primas, ddof=1):.2f}")


Desviación estándar (muestral, ddof=1): 167.57 €
Desviación estándar (poblacional, ddof=0): 167.54 €
Varianza muestral:                          28079.55


### Asimetría (*skewness*)

`scipy.stats.skew(..., bias=False)` calcula la asimetría con corrección de sesgo (recomendada para muestras, igual que `ddof=1` en la varianza). Un valor positivo indica cola larga hacia la derecha (valores altos poco frecuentes tirando de la media hacia arriba) — exactamente lo que cabe esperar de los importes de siniestro.


In [9]:
print(f"Skewness de la prima anual:        {scipy.stats.skew(primas, bias=False):.3f}")
print(f"Skewness del importe de siniestro:  {scipy.stats.skew(importes_siniestros, bias=False):.3f}")


Skewness de la prima anual:        1.435
Skewness del importe de siniestro:  6.908


El importe de siniestro tiene una asimetría claramente más alta y positiva que la prima anual — coherente con lo que ya visteis arriba (media muy por encima de la mediana): unos pocos siniestros muy severos estiran la distribución hacia la derecha. Es el mismo patrón que justifica, en la práctica actuarial, no asumir normalidad para la severidad sin comprobarlo antes (lo retomamos en el notebook de distribuciones).

### Percentiles


In [10]:
percentiles_siniestros = np.percentile(importes_siniestros, [25, 50, 75, 95, 99])
for p, valor in zip([25, 50, 75, 95, 99], percentiles_siniestros):
    print(f"Percentil {p:>2}: {valor:>10,.2f} €")


Percentil 25:     191.94 €
Percentil 50:     703.99 €
Percentil 75:   2,464.40 €
Percentil 95:   8,309.48 €
Percentil 99:  17,630.38 €


El percentil 99 de los importes de siniestro os dice el umbral que solo supera el 1% de los siniestros — una referencia habitual para dimensionar reaseguro de exceso de pérdida o provisiones para grandes siniestros.

**Para ti:** calcula la media, la mediana y el percentil 90 del `importe_pagado` de los siniestros filtrados por `producto == "AUTO"`. Compara la media con la mediana: ¿qué te dice la diferencia sobre la forma de la distribución de severidad en AUTO?


In [11]:
# Tu código aquí


## 3. Correlación

### Pearson: correlación lineal

`scipy.stats.pearsonr(x, y)` mide qué tan cerca está la relación entre `x` e `y` de una línea recta, y devuelve tanto el coeficiente (`r`, entre -1 y 1) como el **p-valor** de la hipótesis nula "la correlación real en la población es 0".

Ejemplo de dominio: ¿existe correlación entre la edad del tomador y la prima anual?


In [12]:
r_pearson = scipy.stats.pearsonr(cartera["edad_tomador"], cartera["prima_anual"])
print(f"Pearson r = {r_pearson.statistic:.3f}, p-valor = {r_pearson.pvalue:.4f}")


Pearson r = 0.155, p-valor = 0.0000


Un `r` cercano a 0 con un p-valor alto indicaría que, en esta cartera sintética, la edad del tomador no está linealmente asociada a la prima anual (razonable: la prima depende sobre todo del producto y otros factores, no directamente de la edad en el generador de datos). Recordad: el p-valor evalúa si el `r` observado es compatible con una correlación poblacional de 0, **no** mide "qué tan fuerte" es la relación — eso lo dice el propio `r`.

### Spearman y Kendall: correlación de rango

Cuando la relación no es necesariamente lineal, o cuando hay valores atípicos que podrían distorsionar Pearson, se recurre a coeficientes basados en el **orden** (rango) de los valores, no en su magnitud exacta.


In [13]:
r_spearman = scipy.stats.spearmanr(cartera["edad_tomador"], cartera["prima_anual"])
tau_kendall = scipy.stats.kendalltau(cartera["edad_tomador"], cartera["prima_anual"])

print(f"Spearman rho = {r_spearman.statistic:.3f}, p-valor = {r_spearman.pvalue:.4f}")
print(f"Kendall tau  = {tau_kendall.statistic:.3f}, p-valor = {tau_kendall.pvalue:.4f}")


Spearman rho = 0.200, p-valor = 0.0000
Kendall tau  = 0.135, p-valor = 0.0000


**Cuándo usar cada uno:**
- **Pearson**: relación lineal, sin valores atípicos extremos.
- **Spearman**: relación monótona pero no necesariamente lineal; más robusto que Pearson ante atípicos.
- **Kendall**: más robusto todavía que Spearman con muestras pequeñas o muchos atípicos, pero más costoso computacionalmente (O(n²) frente a O(n log n) de Spearman) — para datasets muy grandes, Spearman suele ser la opción práctica.

Ejemplo donde sí aparece una relación clara y monótona: la relación entre `franquicia` y `prima_anual` dentro de AUTO (a menor franquicia, mayor prima — el asegurador asume más riesgo, y lo repercute en el precio).


In [14]:
auto = cartera[cartera["producto"] == "AUTO"]

r_pearson_auto = scipy.stats.pearsonr(auto["franquicia"], auto["prima_anual"])
r_spearman_auto = scipy.stats.spearmanr(auto["franquicia"], auto["prima_anual"])

print(f"AUTO -- Pearson r  = {r_pearson_auto.statistic:.3f} (p={r_pearson_auto.pvalue:.4f})")
print(f"AUTO -- Spearman rho = {r_spearman_auto.statistic:.3f} (p={r_spearman_auto.pvalue:.4f})")


AUTO -- Pearson r  = -0.021 (p=0.4455)
AUTO -- Spearman rho = -0.013 (p=0.6219)


> ⚠️ **Error típico**: interpretar un p-valor bajo como "la correlación es fuerte". Un p-valor bajo solo dice que es poco probable observar ese `r` (o uno más extremo) si la correlación poblacional real fuera 0 — con muestras muy grandes, hasta una correlación débil (`r = 0.05`) puede dar un p-valor minúsculo. La **fuerza** de la relación la mide el coeficiente en sí (`r`, `rho`, `tau`), no el p-valor.


## Resumen

- `scipy.stats` complementa a `numpy`: medias no aritméticas (`hmean`, `gmean`), moda (`mode`), asimetría con corrección de sesgo (`skew(..., bias=False)`), y los tres coeficientes de correlación con su p-valor.
- `np.mean()`/`np.var()`/`np.std()` devuelven `nan` si hay valores nulos en el array (usad `np.nanmean()` etc. si queréis ignorarlos) — a diferencia de `pandas`, que los ignora por defecto.
- Para varianza/desviación estándar **muestral** (el caso habitual con datos de cartera), especificad siempre `ddof=1` — por defecto NumPy calcula la versión poblacional (`ddof=0`).
- La media ponderada (`np.average(..., weights=...)`) es la forma correcta de promediar subgrupos de tamaño distinto — comprobadlo siempre contra la media directa sobre el conjunto completo.
- Pearson mide relación lineal; Spearman y Kendall miden relación de rango (monótona), más robustas ante atípicos — Kendall más aún que Spearman, pero más costosa de calcular.
- El p-valor de una correlación evalúa si el coeficiente observado es compatible con una correlación poblacional de 0 — **no** mide la fuerza de la relación, eso lo dice el coeficiente mismo.

**Siguiente:** `05_teoria_distribuciones_probabilidad.ipynb` — distribuciones de probabilidad con `scipy.stats` (normal, binomial, Poisson).
